# Importing libraries

In [1]:
import numpy as np
import pandas as pd

# Reading the data

In [2]:
df= pd.read_csv('Y:/python/data analysis/my datasets/dirty_cafe_sales-tb5ibm.csv')

In [3]:
print(df.head())
print(df.info())
print(df.describe())

  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2              2           4     Credit Card   
1    TXN_4977031    Cake        4              3          12            Cash   
2    TXN_4271903  Cookie        4              1       ERROR     Credit Card   
3    TXN_7034554   Salad        2              5          10         UNKNOWN   
4    TXN_3160411  Coffee        2              2           4  Digital Wallet   

   Location Transaction Date  
0  Takeaway         9/8/2023  
1  In-store        5/16/2023  
2  In-store        7/19/2023  
3   UNKNOWN        4/27/2023  
4  In-store        6/11/2023  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   obj

# Integrating records

In [4]:
df['Item'] = df['Item'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)
# df.loc[df['Item'].isna(), 'Item'] = 'UNKNOWN'
print(df['Item'].unique())

df['Payment Method'] = df['Payment Method'].replace(['UNKNOWN', 'ERROR', 'NA',''], np.nan)
# df.loc[df['Payment Method'].isna(), 'Payment Method'] = 'UNKNOWN'
print(df['Payment Method'].unique())

df['Location'] = df['Location'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)
# df.loc[df['Location'].isna(), 'Location'] = 'UNKNOWN'
print(df['Location'].unique())
print(df.info())

['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' nan 'Sandwich' 'Juice' 'Tea']
['Credit Card' 'Cash' nan 'Digital Wallet']
['Takeaway' 'In-store' nan]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9031 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    6822 non-null   object
 6   Location          6039 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB
None


# price per unit cleaning

filling missing data for price per unit

In [5]:
price_dict={
    'Coffee':2,
    'Cake':3,
    'Cookie':1,
    'Salad':5,
    'Smoothie':4,
    'Sandwich':4,
    'Juice':3,
    'Tea':1.5
}
df['Price Per Unit'] = df['Price Per Unit'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)
# changing data type for future calculations
df['Price Per Unit'] = df['Price Per Unit'].astype(np.float64)

df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].map(price_dict))
print(df['Price Per Unit'].describe())
print(df['Price Per Unit'].unique())

count    9946.000000
mean        2.947667
std         1.280653
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: Price Per Unit, dtype: float64
[2.  3.  1.  5.  4.  1.5 nan]


# quantity and total spent cleaning

changing data type

In [6]:
df['Quantity'] = df['Quantity'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)
df['Total Spent'] = df['Total Spent'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)

df['Quantity'] = df['Quantity'].round(0).astype(pd.Int64Dtype())
df['Total Spent'] = df['Total Spent'].astype(np.float64)

print(df['Quantity'].describe())
print(df['Total Spent'].describe())

count      9521.0
mean     3.028463
std      1.419007
min           1.0
25%           2.0
50%           3.0
75%           4.0
max           5.0
Name: Quantity, dtype: Float64
count    9498.000000
mean        8.924352
std         6.009919
min         1.000000
25%         4.000000
50%         8.000000
75%        12.000000
max        25.000000
Name: Total Spent, dtype: float64


filling missing values

In [7]:
#total spent
mask1 = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[mask1, 'Total Spent'] = df['Quantity'] * df['Price Per Unit']

#quantity
mask2 = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna()
df.loc[mask2, 'Quantity'] = df['Total Spent'] / df['Price Per Unit']

#price per unit
missing_price_mask = df['Price Per Unit'].isna() & df['Quantity'].notna() & df['Total Spent'].notna()
df.loc[missing_price_mask, 'Price Per Unit'] = df['Total Spent'] / df['Quantity']

df['Total Spent'] = df['Total Spent'].round(2)
df['Price Per Unit'] = df['Price Per Unit'].round(2)


print(df['Total Spent'].describe())
print(df['Quantity'].describe())
print(df['Price Per Unit'].describe())

count    9977.000000
mean        8.930139
std         6.004921
min         1.000000
25%         4.000000
50%         8.000000
75%        12.000000
max        25.000000
Name: Total Spent, dtype: float64
count      9977.0
mean     3.024957
std      1.420395
min           1.0
25%           2.0
50%           3.0
75%           4.0
max           5.0
Name: Quantity, dtype: Float64
count    9994.000000
mean        2.947018
std         1.280006
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         5.000000
Name: Price Per Unit, dtype: float64



missing values dropped considerably!!!

# filling more uknown values

In [8]:
print(df['Item'].describe())

count      9031
unique        8
top       Juice
freq       1171
Name: Item, dtype: object


In [9]:
df.loc[df['Price Per Unit'] == 1.5, 'Item'] = 'Tea'
df.loc[df['Price Per Unit'] == 2, 'Item'] = 'Coffee'
df.loc[df['Price Per Unit'] == 1, 'Item'] = 'Cookie'
df.loc[df['Price Per Unit'] == 5, 'Item'] = 'Salad'
print(df['Item'].describe())

count       9520
unique         8
top       Coffee
freq        1291
Name: Item, dtype: object


# transaction date cleaning

In [10]:
df['Transaction Date'] = df['Transaction Date'].replace(['UNKNOWN', 'ERROR', 'N/A',''], 'UNKNOWN')
df.loc[df['Transaction Date'].isna(), 'Transaction Date'] = 'UNKNOWN'

unknown_count = (df['Transaction Date'] == 'UNKNOWN').sum()
print(f"Number of UNKNOWN values: {unknown_count}")

Number of UNKNOWN values: 460


In [11]:
total_rows = 10000
missing_dates = 460
missing_percentage = (missing_dates / total_rows) * 100

print(f"Total rows: {total_rows}")
print(f"Missing dates: {missing_dates}")
print(f"Missing percentage: {missing_percentage:.1f}%")
print(f"Remaining rows if delete: {total_rows - missing_dates} ({100-missing_percentage:.1f}%)")

Total rows: 10000
Missing dates: 460
Missing percentage: 4.6%
Remaining rows if delete: 9540 (95.4%)


In [12]:
df['Transaction Date'] = df['Transaction Date'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])
print(df['Transaction Date'].describe())

count                             9540
mean     2023-07-01 23:00:31.698113280
min                2023-01-01 00:00:00
25%                2023-04-01 00:00:00
50%                2023-07-02 00:00:00
75%                2023-10-02 00:00:00
max                2023-12-31 00:00:00
Name: Transaction Date, dtype: object


In [13]:


# Create a flag for missing dates
df['Date missing'] = df['Transaction Date'].isna()

# Keep all data, but can filter when needed
# Analysis without dates (includes all data)
total_revenue = df['Total Spent'].sum()

# Date-specific analysis (excludes missing dates)
date_analysis = df[~df['Date missing']].groupby(pd.Grouper(key='Transaction Date', freq='ME'))['Total Spent'].sum()
print(date_analysis)

Transaction Date
2023-01-31    7254.0
2023-02-28    6644.0
2023-03-31    7216.0
2023-04-30    7179.0
2023-05-31    6957.5
2023-06-30    7353.0
2023-07-31    6877.5
2023-08-31    7112.5
2023-09-30    6871.0
2023-10-31    7314.0
2023-11-30    6967.0
2023-12-31    7177.0
Freq: ME, Name: Total Spent, dtype: float64


# checking all of data

In [14]:
print(f"Duplicate rows: {df.duplicated().sum()}")

Duplicate rows: 0


In [15]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9520 non-null   object        
 2   Quantity          9977 non-null   Int64         
 3   Price Per Unit    9994 non-null   float64       
 4   Total Spent       9977 non-null   float64       
 5   Payment Method    6822 non-null   object        
 6   Location          6039 non-null   object        
 7   Transaction Date  9540 non-null   datetime64[ns]
 8   Date missing      10000 non-null  bool          
dtypes: Int64(1), bool(1), datetime64[ns](1), float64(2), object(4)
memory usage: 644.7+ KB
None


turning objects to category

In [16]:
df['Item'] = df['Item'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)
df['Payment Method'] = df['Payment Method'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)
df['Location'] = df['Location'].replace(['UNKNOWN', 'ERROR', 'N/A',''], np.nan)
df['Item'] = df['Item'].astype('category')
df['Payment Method'] = df['Payment Method'].astype('category')
df['Location'] = df['Location'].astype('category')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  object        
 1   Item              9520 non-null   category      
 2   Quantity          9977 non-null   Int64         
 3   Price Per Unit    9994 non-null   float64       
 4   Total Spent       9977 non-null   float64       
 5   Payment Method    6822 non-null   category      
 6   Location          6039 non-null   category      
 7   Transaction Date  9540 non-null   datetime64[ns]
 8   Date missing      10000 non-null  bool          
dtypes: Int64(1), bool(1), category(3), datetime64[ns](1), float64(2), object(1)
memory usage: 440.2+ KB
None


category data type is better for memory optimization